In [1]:
import sys
from pathlib import Path

try:
    # 일반 Python 스크립트 실행 시 (__file__이 존재)
    current_path = Path(__file__).resolve()
except NameError:
    # Jupyter Notebook 실행 시 (__file__ 없음)
    current_path = Path().resolve()

# 현재 경로에서 stock_forecast 폴더까지 자동 탐색
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

# sys.path에 추가
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

print(f"sys.path에 등록된 경로: {stock_forecast_path}")

sys.path에 등록된 경로: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\stock_forecast


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [3]:
def add_yoy_growth(df, value_column='value', group_column='root_hs_code', date_column='date'):
    """
    root_hs_code별로 value 컬럼의 연간 증가율을 계산하여 새로운 컬럼으로 추가합니다.
    """
    df = df.copy()
    # top_company_codes = avg_yoy_by_code.sort_values(by='avg_forecast_yoy', ascending=False).head(company_num)
    df = df.dropna(axis=0)
    df[date_column] = pd.to_datetime(df[date_column])
    df.sort_values(by=[group_column, date_column], inplace=True)

    # YoY (12개월 전 대비 비율 변화율) 계산
    df[f'{value_column}_yoy'] = (
        df.groupby(group_column)[value_column]
        .transform(lambda x: x.pct_change(periods=12))
    )

    return df

In [4]:
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host' : '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

trade_df = fetch_table_data(db_info, 'korea_monthly_trade_data_forecast')

import pandas as pd

# date를 datetime으로 변환
trade_df['date'] = pd.to_datetime(trade_df['date'])

# 결측치 제거
trade_df = trade_df.dropna(subset=['expDlr_forecast_12m'])

# root_hs_code, date로 정렬
trade_df = trade_df.sort_values(['root_hs_code', 'date'])

# 그룹 연산 준비
grouped = trade_df.groupby('root_hs_code')

# 이전 12개월 합계 (trailing)
trade_df['export_trail_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum()
)

# 이후 12개월 합계 (forward)
# shift(-11)은 앞으로 11개월 밀어 rolling 12로 보면 해당 시점 기준 이후 12개월을 의미
trade_df['export_forward_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.shift(-11).rolling(window=12, min_periods=12).sum()
)

# YoY 성장률
trade_df['export_yoy_growth'] = (
    (trade_df['export_forward_12m'] / trade_df['export_trail_12m']) - 1
)

# 필요한 컬럼만 보기
result_df = trade_df[['date', 'root_hs_code',
                      'export_trail_12m', 'export_forward_12m', 'export_yoy_growth']]

# 필요시 최근 데이터만
trade_yoy_growth = result_df[result_df['date'] == '2025-06-30']

# 확인
# print(trade_yoy_growth.head(20))


✅ 'korea_monthly_trade_data_forecast' 테이블에서 241385건의 데이터를 가져왔습니다.


In [5]:
len(trade_yoy_growth['root_hs_code'].unique().tolist())

1020

In [ ]:
# ticker  ='A353200'
# hscd = ''

In [6]:
trade_yoy_growth[trade_yoy_growth['root_hs_code'] == '854232']

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
221625,2025-06-30,854232,7.638940e+10,9.837231e+10,0.287774


In [7]:
company_df = fetch_table_data(db_info, 'korea_company_hscode_map')
company_df = company_df.rename(columns={'hs_code' : 'root_hs_code'})
# company_df.rename(columns={'hs_code_6d': 'root_hs_code'}, inplace=True)

✅ 'korea_company_hscode_map' 테이블에서 762건의 데이터를 가져왔습니다.


In [8]:
company_df['root_hs_code'] = company_df['root_hs_code'].astype(str)
trade_yoy_growth['root_hs_code'] = trade_yoy_growth['root_hs_code'].astype(str)

monster_df = pd.merge(company_df, trade_yoy_growth, on='root_hs_code', how='left', indicator=True)

In [9]:
monster_df

,ticker,Name,root_hs_code,date,export_trail_12m,export_forward_12m,export_yoy_growth,_merge
0,A093370,후성,854321,NaT,NaN,NaN,NaN,left_only
1,A036490,SK머티리얼즈,281290,2025-06-30,1.645956e+08,1.932857e+08,0.174307,both
2,A036490,SK머티리얼즈,8542,2025-06-30,1.245768e+11,1.397260e+11,0.121605,both
3,A104830,원익머트리얼즈,854239,2025-06-30,1.145328e+10,1.100684e+10,-0.038979,both
4,A144960,뉴파워프라즈마,854239,2025-06-30,1.145328e+10,1.100684e+10,-0.038979,both
...,...,...,...,...,...,...,...,...
757,A011784,금호석유,4002590000,2025-06-30,2.590850e+08,2.658352e+08,0.026054,both
758,A011785,금호석유,4002110000,2025-06-30,5.967387e+07,7.141741e+07,0.196795,both
759,A178920,PI첨단소재,3916909000,2025-06-30,2.285735e+07,2.496001e+07,0.091991,both
760,A000070,삼양홀딩스,290723,2025-06-30,3.184257e+08,2.657856e+08,-0.165314,both


In [11]:
monster_df[monster_df['Name'].str.contains('삼')]

,ticker,Name,root_hs_code,date,export_trail_12m,export_forward_12m,export_yoy_growth,_merge
57,A005930,삼성전자,8542,2025-06-30,1.245768e+11,1.397260e+11,0.121605,both
59,A006400,삼성SDI,850760,2025-06-30,4.918601e+09,3.884965e+09,-0.210148,both
61,A009150,삼성전기,853224,2025-06-30,1.304833e+09,1.526715e+09,0.170046,both
86,A053700,삼보모터스,870899,2025-06-30,6.878595e+09,6.716661e+09,-0.023542,both
115,A023000,삼원강재,820900,2025-06-30,3.737744e+08,3.657592e+08,-0.021444,both
136,A004380,삼익THK,841340,2025-06-30,1.373462e+08,1.419864e+08,0.033785,both
137,A004380,삼익THK,848210,2025-06-30,3.668347e+08,3.884109e+08,0.058817,both
199,A000390,삼화페인트,391810,2025-06-30,7.709883e+08,7.752429e+08,0.005518,both
250,A009620,삼보산업,870324,2025-06-30,8.377187e+09,7.438424e+09,-0.112062,both
299,A001820,삼화콘덴서,850440,2025-06-30,9.605826e+08,1.059274e+09,0.102741,both
